# HW3 — Human / AI / Fusion text classification

All logic is in `hw3_text_detection.py`; this notebook runs the pipeline stage by stage.
Runs on **Google Colab** and **locally** (VS Code) without changes: paths and device are detected automatically.

Heavy stages save their results to `runs/<name>/` on Google Drive. Set a toggle to `False`
to skip a stage and load its saved results instead.

## 0. Setup

In [ ]:
# ---------------- toggles ----------------
RUN_EDA = True
RECOMPUTE_GROUPS = False   # cached in splits/groups.csv
RECOMPUTE_SPLIT = False    # cached in splits/split_v1.csv (always recomputed with new groups)
RUN_BASELINE = True        # False -> load runs/baseline_tfidf
RUN_ENCODER = True         # False -> load runs/encoder_xlmr_chunks

SUBSAMPLE_N = 40_000       # fixed subsample of whole groups; 20% of it is validation
BASELINE_RUN = "baseline_tfidf"
ENCODER_RUN = "encoder_xlmr_chunks"

# ---------------- environment ----------------
REPO_URL = "https://github.com/Rein-h/pavlunyshyn_hw3_AI.git"
LOCAL_ROOT = r"G:\My Drive\ucu-hw3"                    # local run only: folder that contains data/

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if os.path.exists(REPO_DIR):
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    %pip install -q -r {REPO_DIR}/requirements.txt
    sys.path.insert(0, REPO_DIR)

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import hw3_text_detection as hw

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
hw.seed_everything(hw.SEED)
RECOMPUTE_SPLIT = RECOMPUTE_SPLIT or RECOMPUTE_GROUPS

ENC_CFG = hw.EncoderConfig(
    model_name="xlm-roberta-base",
    max_len=512,          # tokens per window
    overlap=64,           # overlap of neighbouring windows
    train_windows=1,      # random windows per text per epoch
    max_eval_windows=8,   # windows averaged per text at inference
    epochs=2,
    lr=2e-5,
)

paths = hw.get_paths(local_root=LOCAL_ROOT)
hw.describe_environment()

## 1. Data

In [ ]:
train, test, sample = hw.load_data(paths)
hw.data_overview(train, test, sample)

## 2. EDA
### 2.1 Class balance by language and domain

In [ ]:
if RUN_EDA:
    tables = hw.class_balance_tables(train)
    display(tables["label"], tables["lang x label"], tables["domain x label"])
    hw.plot_class_balance(train)
    plt.show()

### 2.2 Text lengths

In [ ]:
if RUN_EDA:
    hw.plot_lengths(train)
    plt.show()
    display(hw.length_table(train))

### 2.3 Length in encoder tokens
Share of texts that do not fit into one encoder window — the reason the encoder works on windows (chunks).

In [ ]:
if RUN_EDA:
    display(hw.token_length_stats(train["text"], ENC_CFG.model_name, ENC_CFG.max_len))

### 2.4 Examples of each class

In [ ]:
if RUN_EDA:
    hw.show_examples(train, n_chars=300)

### 2.5 Train vs. test distribution

In [ ]:
if RUN_EDA:
    display(hw.compare_train_test(train, test))

## 3. Groups of versions and the fixed validation split
A human text and its ai / fusion versions are near-duplicates and are always on the same side of the
train/test split. Groups are rebuilt from shared word 5-grams; the validation split keeps every group
whole, so validation mimics the test split.

In [ ]:
train["group"] = hw.load_or_make_groups(train, paths, recompute=RECOMPUTE_GROUPS)
hw.group_composition(train)

In [ ]:
tr, va = hw.load_or_make_split(train, paths, n_samples=SUBSAMPLE_N, recompute=RECOMPUTE_SPLIT)
hw.split_summary(tr, va)

### 3.1 What makes fusion different
Every ai / fusion text that has a human version in its group is compared with that human version:
overlap of word 3-grams, share of sentences copied verbatim, relative length.

In [ ]:
if RUN_EDA:
    cmp = hw.compare_with_human_version(train)
    display(cmp.groupby("label").median().round(3))
    hw.plot_human_version_comparison(cmp)
    plt.show()
    hw.show_group_example(train, n_chars=400)

## 4. Baseline: TF-IDF + linear models
TF-IDF is fitted on the train split only. Small grid search (feature set × model × C) on the fixed validation split.

In [ ]:
BL_DIR = paths.runs / BASELINE_RUN
if RUN_BASELINE:
    bl_grid, bl = hw.run_baseline(tr, va, test)
    bl_sub = hw.make_submission(test["id"], np.array(hw.LABELS)[bl["test_proba"].argmax(1)], sample)
    hw.save_run(BL_DIR, bl["metrics"], bl["config"], va["id"], bl["val_proba"],
                test["id"], bl["test_proba"], bl_sub)
    bl_grid.to_csv(BL_DIR / "grid.csv", index=False)

bl_run = hw.load_run(BL_DIR)
pd.read_csv(BL_DIR / "grid.csv")

In [ ]:
bl_per_class, bl_by, _ = hw.run_report(bl_run, va, f"Baseline {bl_run['config']}")
plt.show()
display(bl_per_class, bl_by)

## 5. Encoder: fine-tuned multilingual transformer
Long texts are split into overlapping windows of `max_len` tokens. Training sees a random window of
each text (a new one every epoch); prediction averages the probabilities of up to `max_eval_windows`
windows that cover the text. The best epoch is selected on the same validation split as the baseline.

If the Colab session dies, rerun this cell: training resumes from the last finished epoch.

In [ ]:
ENC_DIR = paths.runs / ENCODER_RUN
if RUN_ENCODER:
    enc = hw.train_encoder(tr, va, test, ENC_CFG, ENC_DIR)
    enc_sub = hw.make_submission(test["id"], np.array(hw.LABELS)[enc["test_proba"].argmax(1)], sample)
    hw.save_run(ENC_DIR, enc["metrics"], enc["config"], va["id"], enc["val_proba"],
                test["id"], enc["test_proba"], enc_sub)

enc_run = hw.load_run(ENC_DIR)
pd.DataFrame(enc_run["metrics"]["history"])

In [ ]:
enc_per_class, enc_by, _ = hw.run_report(enc_run, va, f"Encoder {ENC_CFG.model_name}")
plt.show()
display(enc_per_class, enc_by)

## 6. Baseline vs. encoder (same validation split)
Submission files: `runs/<run name>/submission.csv` on Google Drive.

In [ ]:
hw.compare_runs(paths, [BASELINE_RUN, ENCODER_RUN], va)